# Spatial Phenotype Enrichment

This notebook extends the spatial phenotype interaction summary by adding abundance-aware normalization. Raw neighbor counts show how often two phenotypes are adjacent, but raw counts are strongly influenced by how common each phenotype is. A common phenotype can have many neighbor edges simply because many cells of that phenotype exist.

Spatial enrichment asks a more specific question:

```text
Is a phenotype pair observed more often than expected from phenotype abundance?
```

Main inputs:

```text
results/13_cells_with_phenotypes.csv
results/17_spatial_phenotype_interactions_by_image.csv
results/18_spatial_phenotype_interactions_by_category.csv
```

Main outputs:

```text
results/20_spatial_phenotype_enrichment_by_image.csv
results/21_spatial_phenotype_enrichment_by_category.csv
figures/07_spatial_phenotype_enrichment_heatmap.png
```

Important interpretation boundary:

- This step uses rule-based approximate phenotypes.
- Enrichment values describe spatial adjacency relative to abundance.
- Enrichment does not prove direct biological communication.
- Formal statistical testing or permutation analysis is required before making strong biological claims.


### Spatial Phenotype Enrichment (Explained)

#### Overview
Spatial phenotype enrichment extends raw neighbor interaction analysis by accounting for phenotype abundance. Raw neighbor counts are influenced by how common each phenotype is. Enrichment normalizes these counts to determine whether two phenotypes are adjacent more often than expected by chance.

#### Key Question
Is a phenotype pair observed more frequently than expected based on their abundance?

#### Core Concept

| Quantity | Meaning |
|----------|--------|
| Observed edges | Actual number of neighboring cell pairs |
| Expected edges | Number of edges expected based on phenotype abundance |
| Enrichment | Ratio of observed to expected edges |

#### Formula
```text
enrichment = observed_edges / expected_edges
```

#### Interpretation

| Enrichment Value | Meaning |
|------------------|--------|
| Greater than 1 | Enriched, cells are closer than expected |
| Equal to 1 | Random spatial distribution |
| Less than 1 | Depleted, cells avoid each other |

#### Intuition

| Concept | Explanation |
|--------|------------|
| Raw counts | Reflect how common a phenotype is |
| Enrichment | Reflects spatial preference independent of abundance |

---

### How Expected Edges Are Calculated

#### Step 1: Phenotype Abundance

From the cell table, compute fraction of each phenotype:

| Phenotype | Fraction |
|----------|----------|
| Plasma_cell_like | 0.40 |
| T_cell_like | 0.10 |
| Adipocyte_like | 0.05 |

#### Step 2: Assume Random Mixing
Cells are assumed to be randomly distributed. Expected interactions depend only on phenotype abundance.

#### Step 3: Compute Expected Probabilities

**Case1: same phenotype interactions:**

```text
expected_prob = fraction(A) × fraction(A)
```
Example:

Plasma = 0.40

expected = 0.40 × 0.40 = 0.16


| Phenotype | Fraction | Expected Probability |
|----------|----------|----------------------|
| Plasma | 0.40 | 0.16 |

**Case 2: different phenotype interactions:**

```text
expected_prob = 2 × fraction(A) × fraction(B)
```
Example:

Plasma = 0.40, 
T cell = 0.10

expected = 2 × 0.40 × 0.10 = 0.08


| Phenotype A | Phenotype B | Expected Probability |
|-------------|-------------|----------------------|
| Plasma | T cell | 0.08 |

#### Step 4: Convert to Expected Edge Counts

```text
expected_edges = expected_prob × total_edges_in_image
```
Example: If total edges = 10,000,

expected_edges (Plasma–Plasma) = 0.16 × 10000 = 1600

expected_edges (Plasma–T) = 0.08 × 10000 = 800

| Interaction | Expected Probability | Total Edges | Expected Edges |
|------------|----------------------|-------------|----------------|
| Plasma–Plasma | 0.16 | 10000 | 1600 |
| Plasma–T cell | 0.08 | 10000 | 800 |

#### Step 5: Compare with Observed

Example: if observed_edges (Plasma–Plasma) = 2500,

| Interaction | Observed Edges | Expected Edges | Enrichment |
|------------|----------------|----------------|------------|
| Plasma–Plasma | 2500 | 1600 | 2500/1600 = 1.56 |

#### Interpretation

| Value | Meaning |
|------|--------|
| Greater than 1 | Cells cluster together more than expected |
| Equal to 1 | Random interaction |
| Less than 1 | Cells avoid each other |

#### Key Insight

| Concept | Explanation |
|--------|------------|
| Expected edges | Based on abundance and random mixing |
| Observed edges | From spatial neighbor analysis |
| Enrichment | Measures spatial preference |

#### Important Notes

| Concept | Limitation |
|--------|------------|
| Rule-based phenotypes | Approximate cell identities |
| Enrichment | Indicates proximity, not communication |
| Statistical validation | Requires permutation testing for strong conclusions |

## Step 0: Configure Workflow Paths

This setup cell defines the input and output paths for the enrichment analysis.

Required inputs:

- The phenotyped cell table provides phenotype abundance by image and category.
- The image-level interaction table provides observed phenotype-neighbor edge counts per ROI.
- The category-level interaction table provides observed phenotype-neighbor edge counts per sample category.

The enrichment script combines these inputs to calculate expected edge fractions from phenotype abundance.


In [1]:
from pathlib import Path
import csv
import subprocess

current = Path.cwd().resolve()
if current.name == 'notebooks':
    WORKFLOW_DIR = current.parent
else:
    WORKFLOW_DIR = current

RESULTS_DIR = WORKFLOW_DIR / 'results'
FIGURES_DIR = WORKFLOW_DIR / 'figures'
SCRIPTS_DIR = WORKFLOW_DIR / 'scripts'

PHENOTYPED_CELLS = RESULTS_DIR / '13_cells_with_phenotypes.csv'
INTERACTIONS_IMAGE = RESULTS_DIR / '17_spatial_phenotype_interactions_by_image.csv'
INTERACTIONS_CATEGORY = RESULTS_DIR / '18_spatial_phenotype_interactions_by_category.csv'

print('Workflow directory:', WORKFLOW_DIR)
print('Phenotyped cell table exists:', PHENOTYPED_CELLS.exists())
print('Image-level interaction table exists:', INTERACTIONS_IMAGE.exists())
print('Category-level interaction table exists:', INTERACTIONS_CATEGORY.exists())
print('Enrichment script exists:', (SCRIPTS_DIR / '12_spatial_phenotype_enrichment.py').exists())


Workflow directory: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction
Phenotyped cell table exists: True
Image-level interaction table exists: True
Category-level interaction table exists: True
Enrichment script exists: True


## Step 1: Understand Observed and Expected Spatial Interactions

The previous notebook counted observed phenotype-neighbor edges. For example, a row may report the number of `Plasma_cell_like -- T_cell_like` neighbor edges in one ROI image.

Raw observed edge count:

```text
number of neighbor edges for a phenotype pair
```

Observed edge fraction:

```text
edge_count / all phenotype-neighbor edges in the image or category
```

Expected edge fraction from abundance:

The expected edge fraction is calculated from phenotype frequencies in the same image or category. If phenotype A represents 20% of cells and phenotype B represents 10% of cells, then a mixed A-B edge is expected at approximately:

```text
2 * 0.20 * 0.10 = 0.04
```

The factor of 2 is used for mixed phenotype pairs because an undirected edge can contain A-B or B-A ordering. For same-phenotype pairs, the expected fraction is:

```text
p(A) * p(A)
```

This expected value is a simple abundance-based baseline. It does not account for tissue architecture, cell size, segmentation shape, or local density differences.


## Step 2: Understand Enrichment Metrics

The script reports several metrics for each phenotype pair.

Columns in the enrichment outputs:

```text
group: Image or category
phenotype_a: first phenotype in the pair
phenotype_b: second phenotype in the pair
edge_count: observed unique undirected neighbor edges
observed_edge_fraction: observed fraction of all edges in the group
expected_edge_fraction_from_abundance: expected fraction based on phenotype abundance
observed_expected_ratio: observed fraction divided by expected fraction
log2_enrichment: log2(observed / expected)
group_cell_count: total cells in the image or category
phenotype_a_cell_count: number of cells with phenotype_a
phenotype_b_cell_count: number of cells with phenotype_b
```

How to interpret `observed_expected_ratio`:

```text
> 1: observed more often than expected from abundance
= 1: observed approximately as expected from abundance
< 1: observed less often than expected from abundance
```

How to interpret `log2_enrichment`:

```text
positive value: enriched adjacency
zero: approximately expected adjacency
negative value: depleted adjacency
```

Example:

```text
log2_enrichment = 1 means observed/expected = 2-fold enrichment
log2_enrichment = -1 means observed/expected = 2-fold depletion
```


## Step 3: Run Spatial Phenotype Enrichment

The script performs the following operations:

```text
1. Read the phenotyped single-cell table.
2. Count phenotype abundance within each image and category.
3. Read observed phenotype-neighbor interactions by image and category.
4. Calculate expected edge fractions from phenotype abundance.
5. Calculate observed/expected ratios.
6. Calculate log2 enrichment values.
7. Save image-level and category-level enrichment tables.
8. Save a heatmap of spatial phenotype enrichment.
```

The image-level table is useful for ROI-specific review. The category-level table is useful for compact exploratory comparison across `MGUS`, `UB`, and `B`.


In [2]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '12_spatial_phenotype_enrichment.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--cells', 'results/13_cells_with_phenotypes.csv',
    '--interactions-image', 'results/17_spatial_phenotype_interactions_by_image.csv',
    '--interactions-category', 'results/18_spatial_phenotype_interactions_by_category.csv',
    '--output-image', 'results/20_spatial_phenotype_enrichment_by_image.csv',
    '--output-category', 'results/21_spatial_phenotype_enrichment_by_category.csv',
    '--heatmap', 'figures/07_spatial_phenotype_enrichment_heatmap.png',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)


Image-level enrichment rows: 732
Category-level enrichment rows: 296
Phenotypes represented: 14
Image-level enrichment table: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/20_spatial_phenotype_enrichment_by_image.csv
Category-level enrichment table: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/21_spatial_phenotype_enrichment_by_category.csv
Enrichment heatmap: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/figures/07_spatial_phenotype_enrichment_heatmap.png



## Step 4: Review Image-Level Enrichment

The image-level enrichment table contains one row per observed phenotype pair in each ROI image.

This table is useful for identifying ROI-specific spatial organization. For example, a phenotype pair may be enriched in one ROI but not in another. Such differences can reflect true tissue heterogeneity, differences in cell abundance, segmentation effects, or conservative phenotype rules.

Interpretation should focus on both the enrichment value and the supporting edge count. High enrichment based on very few edges is less stable than enrichment supported by many edges.


In [3]:
image_enrichment = RESULTS_DIR / '20_spatial_phenotype_enrichment_by_image.csv'
with image_enrichment.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)

for row in rows[:30]:
    print(row)


{'Image': 'TS-373_IMC01_UB_001', 'phenotype_a': 'Adipocyte_like', 'phenotype_b': 'Adipocyte_like', 'edge_count': '120', 'observed_edge_fraction': '0.008659258190215038', 'expected_edge_fraction_from_abundance': '0.0034333085740019626', 'observed_expected_ratio': '2.5221322242298654', 'log2_enrichment': '1.3346439117645839', 'group_cell_count': '5649', 'phenotype_a_cell_count': '331', 'phenotype_b_cell_count': '331'}
{'Image': 'TS-373_IMC01_UB_001', 'phenotype_a': 'Adipocyte_like', 'phenotype_b': 'CD34_positive_cell_like', 'edge_count': '128', 'observed_edge_fraction': '0.009236542069562707', 'expected_edge_fraction_from_abundance': '0.010517748924586073', 'observed_expected_ratio': '0.8781862103564344', 'log2_enrichment': '-0.18740121404005752', 'group_cell_count': '5649', 'phenotype_a_cell_count': '331', 'phenotype_b_cell_count': '507'}
{'Image': 'TS-373_IMC01_UB_001', 'phenotype_a': 'Adipocyte_like', 'phenotype_b': 'CD4_T_cell_like', 'edge_count': '28', 'observed_edge_fraction': '0.0

## Step 5: Review Category-Level Enrichment

The category-level enrichment table aggregates observed interactions and abundance baselines by category.

This table provides a compact exploratory overview of phenotype-neighbor enrichment across `MGUS`, `UB`, and `B`. It should be interpreted cautiously because the representative subset is small and phenotype labels are rule-based approximations.

Useful review approach:

- Sort by high positive `log2_enrichment` to find enriched phenotype pairs.
- Sort by negative `log2_enrichment` to find depleted phenotype pairs.
- Check `edge_count` to avoid over-interpreting rare pairs.
- Compare enrichment with raw composition and interaction tables.


In [4]:
category_enrichment = RESULTS_DIR / '21_spatial_phenotype_enrichment_by_category.csv'
with category_enrichment.open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)

for row in rows[:40]:
    print(row)


{'category': 'B', 'phenotype_a': 'Adipocyte_like', 'phenotype_b': 'Adipocyte_like', 'edge_count': '300', 'observed_edge_fraction': '0.0062996094242156984', 'expected_edge_fraction_from_abundance': '0.0012374605593216145', 'observed_expected_ratio': '5.090755722888812', 'log2_enrichment': '2.347879840335173', 'group_cell_count': '17966', 'phenotype_a_cell_count': '632', 'phenotype_b_cell_count': '632'}
{'category': 'B', 'phenotype_a': 'Adipocyte_like', 'phenotype_b': 'CD34_positive_cell_like', 'edge_count': '49', 'observed_edge_fraction': '0.0010289362059552308', 'expected_edge_fraction_from_abundance': '0.0020206634449682057', 'observed_expected_ratio': '0.5092071163644081', 'log2_enrichment': '-0.9736755132916509', 'group_cell_count': '17966', 'phenotype_a_cell_count': '632', 'phenotype_b_cell_count': '516'}
{'category': 'B', 'phenotype_a': 'Adipocyte_like', 'phenotype_b': 'CD4_T_cell_like', 'edge_count': '43', 'observed_edge_fraction': '0.0009029440174709168', 'expected_edge_fraction

## Step 6: Review Enrichment Heatmap

The heatmap summarizes log2 enrichment values across phenotype pairs. Red-blue style coloring is used so enriched and depleted relationships are visually separated.

Interpretation guidance:

- Positive values indicate more neighbor edges than expected from phenotype abundance.
- Negative values indicate fewer neighbor edges than expected from phenotype abundance.
- Values near zero indicate approximately abundance-expected adjacency.
- Diagonal values describe same-phenotype adjacency.
- Off-diagonal values describe mixed-phenotype adjacency.

Important limitation:

This heatmap is an abundance-normalized exploratory summary, not a formal spatial statistics test. Strong biological interpretation requires validation with robust phenotype annotation, larger sample context, and preferably permutation-based spatial enrichment testing.


In [5]:
heatmap = FIGURES_DIR / '07_spatial_phenotype_enrichment_heatmap.png'
print('Heatmap exists:', heatmap.exists())
print('Heatmap size_bytes:', heatmap.stat().st_size if heatmap.exists() else 0)


Heatmap exists: True
Heatmap size_bytes: 120049


## ** Interpretation of Results **

### Spatial Phenotype Enrichment by Image

#### Overview
`20_spatial_phenotype_enrichment_by_image.csv` quantifies spatial interactions between phenotypes while accounting for their abundance. It compares observed neighbor interactions with expected interactions under random mixing to identify enrichment or depletion.

#### Column Description

| Column | Meaning |
|--------|--------|
| phenotype_a | First cell type |
| phenotype_b | Neighbor cell type |
| edge_count | Actual number of spatial neighbor connections |
| observed_edge_fraction | Proportion of this interaction among all edges |
| expected_edge_fraction_from_abundance | Expected proportion based on phenotype abundance |
| observed_expected_ratio | Ratio of observed to expected interactions |
| log2_enrichment | Log2-transformed enrichment value |
| group_cell_count | Total number of cells in the image |
| phenotype_a_cell_count | Number of cells of phenotype A |
| phenotype_b_cell_count | Number of cells of phenotype B |

---

#### Key Metrics

| Metric | Meaning |
|--------|--------|
| Observed | What is actually measured in the data |
| Expected | What is expected if cells were randomly distributed |
| Ratio | Observed divided by expected |
| Log2 enrichment | Final normalized enrichment score |

#### Formula

$$
\text{observed\_expected\_ratio} = \frac{\text{observed\_edge\_fraction}}{\text{expected\_edge\_fraction}}
$$

$$
\text{log2\_enrichment} = \log_2(\text{observed\_expected\_ratio})
$$

---

#### Example 1: Adipocyte_like -- Adipocyte_like

| Metric | Value |
|--------|------|
| observed_edge_fraction | 0.00866 |
| expected_edge_fraction | 0.00343 |
| observed_expected_ratio | 2.52 |
| log2_enrichment | 1.33 |

Interpretation  
Adipocyte-like cells are approximately 2.5 times more likely to be adjacent to each other than expected by chance, indicating strong spatial clustering.

---

#### Example 2: Adipocyte_like -- CD34_positive_cell_like

| Metric | Value |
|--------|------|
| observed_expected_ratio | 0.88 |
| log2_enrichment | -0.19 |

Interpretation  
This interaction occurs slightly less frequently than expected, indicating no strong spatial association.

---

#### Example 3: CD4_T_cell_like -- CD4_T_cell_like

| Metric | Value |
|--------|------|
| observed_expected_ratio | 7.80 |
| log2_enrichment | 2.96 |

Interpretation  
CD4 T cells show strong clustering, with interactions occurring far more frequently than expected.

---

#### Interpretation of log2_enrichment

| Value | Meaning |
|-------|--------|
| Greater than 1 | Strong enrichment |
| Equal to 0 | Random interaction |
| Less than 0 | Depletion |
| Less than -1 | Strong avoidance |

---

#### Why Log2 Transformation

| Reason | Explanation |
|--------|------------|
| Symmetry | Equal scaling for enrichment and depletion |
| Interpretability | Fold changes become easier to compare |
| Visualization | Suitable for heatmaps |

Example  
2-fold enrichment corresponds to log2 value of 1  
0.5-fold corresponds to log2 value of -1  

---

#### Interpretation Summary

| Metric | Role |
|--------|------|
| edge_count | Raw number of interactions |
| observed_edge_fraction | Normalized interaction frequency |
| expected_edge_fraction | Baseline expectation from abundance |
| observed_expected_ratio | Measure of spatial preference |
| log2_enrichment | Final interpretable score |

---

#### Important Notes

| Concept | Explanation |
|--------|------------|
| Spatial proximity | Indicates physical closeness only |
| Rule-based phenotypes | Approximate cell identities |
| Enrichment values | Reflect relative adjacency, not interaction |
| Statistical validation | Requires permutation testing for robust conclusions |